# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bivo2004/my-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### Task Type: Ranking (via Binary Classification)

While the core prediction is binary (is this page declining: yes or no?), the actual business task is **Ranking**. An editorial team cannot update 10,000 pages at once. The model must output a probability score so we can sort the pages from highest risk to lowest risk, creating a prioritized queue for humans to review.

In [6]:
import os, sys, subprocess
import pandas as pd
import numpy as np

# 1. Setup Colab Environment
if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter"], check=True)
    if not os.getcwd().endswith("flyrank-ml-internship-starter"):
        os.chdir("flyrank-ml-internship-starter")

# 2. Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Data loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns.")

Data loaded successfully: 30000 rows, 44 columns.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target: `is_declining` (Observed Outcome)

We are predicting whether a page is currently bleeding traffic. The label comes from an **observed outcome**, derived directly from the `trend_direction` column (which categorizes historical traffic percentage changes). If `trend_direction == 'down'`, the target label is 1. Otherwise, it is 0.

In [7]:
# Create the binary target label based on the observed trend outcome
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Show the distribution of our target
target_dist = df["is_declining_label"].value_counts(normalize=True) * 100
print("Target Label Distribution:")
print(f"1 (Declining): {target_dist[1]:.1f}%")
print(f"0 (Not Declining): {target_dist[0]:.1f}%")

Target Label Distribution:
1 (Declining): 54.2%
0 (Not Declining): 45.8%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Metric: Precision@K (e.g., Precision@50)

Because editorial bandwidth is highly limited, false positives (rewriting a healthy page) are expensive. We do not need to find *every* declining page (Recall). We need to guarantee that if the model tells a writer to review 50 pages, as close to 100% of them as possible are actually declining. Therefore, **Precision@50** is the metric we must optimize and defend.

In [8]:
# Function to simulate how we will score our model's ranked output
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Let's test what 'good' looks like by scoring a random guess baseline
np.random.seed(42)
random_scores = np.random.rand(len(df))
random_p50 = precision_at_k(random_scores, df["is_declining_label"], 50)

print(f"Random Guess Baseline Precision@50: {random_p50:.3f}")
print("A 'good' ML model needs to significantly beat this baseline (aiming for > 0.700).")

Random Guess Baseline Precision@50: 0.560
A 'good' ML model needs to significantly beat this baseline (aiming for > 0.700).


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### Unit of Analysis: One published web page

One row in our dataset represents a single, unique published URL/article for a specific client. The features describe the page's current state (age, word count) and its recent search performance (impressions, CTR, average position).

In [9]:
# Show a slice of the dataframe proving 1 row = 1 page
unit_slice = df[["content_id", "client_id", "content_type", "trend_direction", "is_declining_label"]].head(3)

print("One row = One piece of content")
display(unit_slice)

One row = One piece of content


,content_id,client_id,content_type,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Complexity of Traffic Decay

Traffic decay is too messy for a simple `if-statement` (e.g., `if age > 180 days -> update`). A page might be old but highly engaged, or brand new but dropping rapidly due to a sudden spike in competition. ML can weigh the non-linear interactions between `age`, `ctr`, `competition`, and `avg_position` simultaneously to find hidden patterns a human rule would miss.

In [10]:
# Show why a simple "stale" if-statement misses things
declining_total = (df["is_declining_label"] == 1).sum()

# Pages that are declining, but were updated recently (less than 6 months ago)
declining_but_fresh = ((df["is_declining_label"] == 1) & (df["days_since_last_update"] < 180)).sum()

print(f"Total declining pages: {declining_total}")
print(f"Declining pages that are LESS than 6 months old: {declining_but_fresh}")
print(f"\nA strict 'only update old pages' rule completely misses {declining_but_fresh/declining_total:.1%} of all decaying content.")
print("ML solves this by finding multi-variable patterns instead of relying on one hard threshold.")

Total declining pages: 16262
Declining pages that are LESS than 6 months old: 16180

A strict 'only update old pages' rule completely misses 99.5% of all decaying content.
ML solves this by finding multi-variable patterns instead of relying on one hard threshold.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.